# EEG Collapse Regime Validation

**Phase-Memory Operator - Deterministic Replay**

Reference: Krüger & Feeney (2026), Section 7 and Appendix B

---

This notebook validates the phase-memory operator on EEG seizure data, treating seizures as **collapse-like regime shifts**. This is a high-dimensional nonstationary dynamical testbed - no biological mechanism is claimed to transfer.

**Protocol**: Calibration → Validation → Test (frozen parameters)

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
from scipy.signal import butter, filtfilt
from sklearn.metrics import roc_curve, auc, precision_recall_curve
import sys
from pathlib import Path

sys.path.insert(0, str(Path.cwd().parent))

from src.phase_extraction import analytic_signal, construct_reference_phase
from src.triadic_operator import compute_triadic_state
from src.stability_functional import (
    compute_stability_functional, calibrate_weights, 
    evaluate_discriminative_power, StabilityParameters
)
from src.warning_logic import (
    detect_warning_events, compute_lead_time, 
    compute_false_alarm_rate, calibrate_alert_threshold, AlertParameters
)

np.random.seed(42)
print("✓ Imports successful")

## 1. Data Generation

> **Note**: This notebook uses synthetic EEG-like data for demonstration. Metrics reported here are illustrative only. The numerical results reported in the paper were obtained using real CHB-MIT EEG data under the same locked parameter protocol.

**Production use**: Replace with actual CHB-MIT EEG loading.

In [ ]:
# Synthetic EEG-like data (replace with real data loading)
def generate_eeg_data(duration=3600, fs=256, n_seizures=3):
    n = int(duration * fs)
    t = np.arange(n) / fs
    
    # Multi-band signal
    signal = (5 * np.sin(2*np.pi*10*t) +  # Alpha
              2 * np.sin(2*np.pi*20*t) +  # Beta  
              3 * np.sin(2*np.pi*6*t))    # Theta
    
    # Pink noise
    noise_fft = np.fft.fft(np.random.randn(n))
    freqs = np.fft.fftfreq(n, 1/fs)
    noise_fft *= 1 / np.sqrt(np.abs(freqs) + 1)
    signal += 2 * np.real(np.fft.ifft(noise_fft))
    
    # Add seizures
    seizure_times = np.linspace(900, duration-300, n_seizures)
    labels = np.zeros(n)
    
    for t_seiz in seizure_times:
        idx = int(t_seiz * fs)
        # Pre-ictal: 60s warning period
        labels[max(0, idx-int(60*fs)):idx] = 1
        # Ictal: 30s seizure
        dur = int(30 * fs)
        t_s = np.arange(dur) / fs
        signal[idx:idx+dur] += 20 * np.sin(2*np.pi*3*t_s) * np.exp(-t_s/10)
        labels[idx:idx+dur] = 1
    
    return signal, t, labels, seizure_times

signal_raw, t, labels, seizure_times = generate_eeg_data()
fs = 256
print(f"✓ Data: {len(t)} samples, {len(seizure_times)} seizures at {seizure_times}s")

In [ ]:
# Preprocessing (logged)
b, a = butter(4, [0.5, 30], btype='bandpass', fs=fs)
signal_filt = filtfilt(b, a, signal_raw)
signal = (signal_filt - signal_filt.mean()) / signal_filt.std()

# Temporal splits
n = len(signal)
cal_end, val_end = int(0.5*n), int(0.75*n)

signal_cal, t_cal, labels_cal = signal[:cal_end], t[:cal_end], labels[:cal_end]
signal_val, t_val, labels_val = signal[cal_end:val_end], t[cal_end:val_end], labels[cal_end:val_end]
signal_test, t_test, labels_test = signal[val_end:], t[val_end:], labels[val_end:]

print(f"✓ Splits: Cal={len(t_cal)} Val={len(t_val)} Test={len(t_test)}")

## 2. Calibration Phase

In [ ]:
# Phase extraction
z_cal, theta_cal, amp_cal = analytic_signal(signal_cal)

# Reference from baseline (first 10 min)
baseline = signal_cal[:int(600*fs)]
theta_ref_base, ref_meta = construct_reference_phase(baseline, dt=1/fs)
omega_ref = ref_meta['omega_ref']
theta_ref_cal = omega_ref * t_cal + theta_ref_base[0]

# Triadic state
state_cal = compute_triadic_state(theta_cal, theta_ref_cal, t_cal)

print(f"✓ Phase extracted, ω_ref={omega_ref:.4f} rad/s")
print(f"  U1: [{state_cal.U1.min():.2f}, {state_cal.U1.max():.2f}]")
print(f"  U2: [{state_cal.U2.min():.2f}, {state_cal.U2.max():.2f}]")
print(f"  U3: [{state_cal.U3.min():.2f}, {state_cal.U3.max():.2f}]")

In [ ]:
# Weight calibration
params, cal_meta = calibrate_weights(
    state_cal.U1, state_cal.U2, state_cal.U3, labels_cal, method='roc_auc'
)

print(f"✓ Weights: a={params.a:.4f}, b={params.b:.4f}, c={params.c:.4f}")
print(f"  Cal AUC: {cal_meta['final_score']:.4f}")

# Alert threshold
J_cal = compute_stability_functional(state_cal.U1, state_cal.U2, state_cal.U3, params)
alert_params, alert_meta = calibrate_alert_threshold(
    J_cal, t_cal, labels_cal, dwell_time=2.0, target_far=0.1
)

print(f"✓ Alert: J_c={alert_params.J_threshold:.4f}, dwell={alert_params.dwell_time}s")
print(f"  FAR: {alert_meta['achieved_far']:.3f}, lead={alert_meta['median_lead_time']:.1f}s")

In [ ]:
# FREEZE PARAMETERS
params.freeze()
alert_params.freeze()
print("✓✓✓ PARAMETERS FROZEN - Cannot be modified ✓✓✓")

## 3. Validation Phase

In [ ]:
# Apply frozen operator to validation
z_val, theta_val, _ = analytic_signal(signal_val)
theta_ref_val = omega_ref * t_val + theta_ref_base[0]
state_val = compute_triadic_state(theta_val, theta_ref_val, t_val)
J_val = compute_stability_functional(state_val.U1, state_val.U2, state_val.U3, params)
warnings_val, _ = detect_warning_events(J_val, t_val, alert_params)

# Metrics
val_seiz = seizure_times[(seizure_times >= t_val[0]) & (seizure_times <= t_val[-1])]
lead_val = compute_lead_time(warnings_val, t_val, val_seiz)
far_val = compute_false_alarm_rate(warnings_val, t_val, val_seiz)

from sklearn.metrics import roc_auc_score
try:
    auc_val = roc_auc_score(labels_val, J_val)
except:
    auc_val = np.nan

print(f"✓ Validation (frozen params):")
print(f"  AUC: {auc_val:.4f}")
print(f"  FAR: {far_val:.3f} alarms/s")
print(f"  Lead times: {[f'{x:.1f}' for x in lead_val if not np.isnan(x)]}s")

## 4. Test Phase (Illustrative Locked Evaluation)

In [ ]:
# Apply frozen operator to TEST (run exactly once)
z_test, theta_test, _ = analytic_signal(signal_test)
theta_ref_test = omega_ref * t_test + theta_ref_base[0]
state_test = compute_triadic_state(theta_test, theta_ref_test, t_test)
J_test = compute_stability_functional(state_test.U1, state_test.U2, state_test.U3, params)
warnings_test, events_test = detect_warning_events(J_test, t_test, alert_params)

# Final metrics
test_seiz = seizure_times[(seizure_times >= t_test[0]) & (seizure_times <= t_test[-1])]
lead_test = compute_lead_time(warnings_test, t_test, test_seiz)
far_test = compute_false_alarm_rate(warnings_test, t_test, test_seiz)

try:
    auc_test = roc_auc_score(labels_test, J_test)
    prec, rec, _ = precision_recall_curve(labels_test, J_test)
    pr_auc_test = auc(rec, prec)
except:
    auc_test, pr_auc_test = np.nan, np.nan

print("="*60)
print("FINAL TEST RESULTS (Locked Parameters)")
print("="*60)
print(f"ROC-AUC:          {auc_test:.4f}")
print(f"PR-AUC:           {pr_auc_test:.4f}")
print(f"False alarm rate: {far_test:.3f} alarms/s")
print(f"Lead times:       {[f'{x:.1f}s' for x in lead_test if not np.isnan(x)]}")
print(f"Detection rate:   {np.sum(~np.isnan(lead_test))/len(lead_test)*100:.1f}%")
print(f"Median lead:      {np.nanmedian(lead_test):.1f}s")
print("="*60)

## 5. Visualization

In [ ]:
# Plot test results
fig, axes = plt.subplots(3, 1, figsize=(14, 10), sharex=True)

# Signal + ground truth
axes[0].plot(t_test, signal_test, 'k', alpha=0.5, linewidth=0.5)
axes[0].fill_between(t_test, -5, 5, where=labels_test>0, alpha=0.2, color='red', label='True events')
axes[0].set_ylabel('Signal')
axes[0].legend()
axes[0].set_title('Test Window: Phase-Memory Operator Performance')

# Stability functional
axes[1].plot(t_test, J_test, 'b', linewidth=1)
axes[1].axhline(alert_params.J_threshold, color='r', linestyle='--', label=f'Threshold={alert_params.J_threshold:.3f}')
axes[1].fill_between(t_test, 0, J_test.max(), where=warnings_test, alpha=0.3, color='orange', label='Warnings')
axes[1].set_ylabel('J(t)')
axes[1].legend()

# Triadic channels
ax3a = axes[2]
ax3a.plot(t_test, state_test.U1, 'b', alpha=0.7, linewidth=0.8, label='U₁')
ax3a.set_ylabel('U₁, U₂', color='b')
ax3a.tick_params(axis='y', labelcolor='b')

ax3b = ax3a.twinx()
ax3b.plot(t_test, state_test.U3, 'r', alpha=0.7, linewidth=0.8, label='U₃')
ax3b.set_ylabel('U₃', color='r')
ax3b.tick_params(axis='y', labelcolor='r')

axes[2].set_xlabel('Time (s)')
axes[2].legend(loc='upper left')
ax3b.legend(loc='upper right')

plt.tight_layout()
plt.savefig('test_results.png', dpi=150)
plt.show()

print("✓ Visualization saved")

In [ ]:
# ROC and PR curves
fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(12, 5))

# ROC
fpr, tpr, _ = roc_curve(labels_test, J_test)
ax1.plot(fpr, tpr, 'b-', linewidth=2, label=f'AUC={auc_test:.3f}')
ax1.plot([0, 1], [0, 1], 'k--', alpha=0.3)
ax1.set_xlabel('False Positive Rate')
ax1.set_ylabel('True Positive Rate')
ax1.set_title('ROC Curve (Test Set)')
ax1.legend()
ax1.grid(alpha=0.3)

# PR
prec, rec, _ = precision_recall_curve(labels_test, J_test)
ax2.plot(rec, prec, 'r-', linewidth=2, label=f'AUC={pr_auc_test:.3f}')
ax2.axhline(np.mean(labels_test), color='k', linestyle='--', alpha=0.3, label='Baseline')
ax2.set_xlabel('Recall')
ax2.set_ylabel('Precision')
ax2.set_title('Precision-Recall Curve (Test Set)')
ax2.legend()
ax2.grid(alpha=0.3)

plt.tight_layout()
plt.show()

## Summary

✅ **Deterministic replay** with frozen parameters  
✅ **Strict temporal splits** (no data leakage)  
✅ **Single test execution** (no parameter tuning on test)  
✅ **Pre-declared metrics** evaluated on locked configuration

> **Note**: Performance metrics shown above are from synthetic data and are illustrative only. Paper results use real CHB-MIT EEG with the same protocol.

**Potential extensions (outside the scope of this repository)**:
1. Cross-validation across multiple CHB-MIT subjects
2. Systematic baseline detector comparisons
3. Multi-channel fusion strategies
4. Perturbation stability analysis

**Full paper**: Krüger & Feeney (2026), "A Deterministic Phase-Memory Operator for Hysteresis and Path-Dependent Regime Shifts"

**Repository**: phase-memory-operator (linked from the paper)